In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


##### Question1: Business city table has data from the day udaan has started operation. 
Write a SQL to identify year-wise count of new cities where udaan started their operations.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
business_city_data = [
    ("2020-01-02", 3),
    ("2020-07-01", 7),
    ("2021-01-01", 3),
    ("2021-02-03", 19),
    ("2022-12-01", 3),
    ("2022-12-15", 3),
    ("2022-02-28", 12)
]
business_city_schema = StructType([
    StructField("business_date", StringType(), True),
    StructField("city_id", IntegerType(), True)
])
business_city_df = spark.createDataFrame(
    business_city_data,
    schema=business_city_schema
)
# Convert string to date
business_city_df = business_city_df.withColumn(
    "business_date",
    to_date("business_date", "yyyy-MM-dd")
)
business_city_df.write.mode("overwrite").saveAsTable("business_city")
business_city_df.orderBy("business_date").display()

business_date,city_id
2020-01-02,3
2020-07-01,7
2021-01-01,3
2021-02-03,19
2022-02-28,12
2022-12-01,3
2022-12-15,3


In [0]:
from pyspark.sql.functions import *
df = business_city_df.groupBy(
    col("city_id")
).agg(
    min(col("business_date")).alias("date")
).withColumn(
    "year",
    year(col("date"))
)

df1 = df.groupBy(
    col("year")
).agg(
    countDistinct(col("city_id")).alias("count")
)

df.display()
df1.display()

city_id,date,year
3,2020-01-02,2020
7,2020-07-01,2020
19,2021-02-03,2021
12,2022-02-28,2022


year,count
2020,2
2021,1
2022,1


##### Question 4: determine phone numbers that satisfy below conditions: 
> - the numbers have both incoming & outgoing calls
> - the sum of duration of outgoing calls should be greater than sum of duration of incoming calls

In [0]:
from pyspark.sql.types import *
call_details_data = [
    ("OUT", "181868", 13),
    ("OUT", "2159010", 8),
    ("OUT", "2159010", 178),
    ("SMS", "4153810", 1),
    ("OUT", "2159010", 152),
    ("OUT", "9140152", 18),
    ("SMS", "4162672", 1),
    ("SMS", "9168204", 1),
    ("OUT", "9168204", 576),
    ("INC", "2159010", 5),
    ("INC", "2159010", 4),
    ("SMS", "2159010", 1),
    ("SMS", "4535614", 1),
    ("OUT", "181868", 20),
    ("INC", "181868", 54),
    ("INC", "218748", 20),
    ("INC", "2159010", 9),
    ("INC", "197432", 66),
    ("SMS", "2159010", 1),
    ("SMS", "4535614", 1)
]
call_details_schema = StructType([
    StructField("call_type", StringType(), True),
    StructField("call_number", StringType(), True),
    StructField("call_duration", IntegerType(), True)
])
call_details_df = spark.createDataFrame(
    call_details_data,
    schema=call_details_schema
)
call_details_df.write.mode("Overwrite").saveAsTable("call_details")
call_details_df.display()

call_type,call_number,call_duration
OUT,181868,13
OUT,2159010,8
OUT,2159010,178
SMS,4153810,1
OUT,2159010,152
OUT,9140152,18
SMS,4162672,1
SMS,9168204,1
OUT,9168204,576
INC,2159010,5


In [0]:

from pyspark.sql.functions import *

df = call_details_df.groupBy(
    col("call_number")
).agg(
    sum(
        when (
            col("call_type") == "OUT",
            col("call_duration")
        )
    ).alias("out_duration"),
    sum(
        when (
            col("call_type") == "INC",
            col("call_duration")
        )
    ).alias("inc_duration")
).filter(
    col("out_duration").isNotNull() & col("inc_duration").isNotNull()
).filter(
    col("out_duration") > col("inc_duration")
)

df.display()

call_number,out_duration,inc_duration
2159010,338,18
